# 3-모델 전체 학습 + 앙상블 노트북
## YOLO11 → YOLO26 → RT-DETR → WBF 앙상블 → submission.csv

**학습 순서**: YOLO11 → YOLO26 → RT-DETR → 앙상블 추론  
**예상 소요 시간 (A100 기준)**:
- YOLO11x : ~2~3시간 (60에폭, imgsz=1280)
- YOLO26 (yolo11x) : ~2~3시간 (60에폭, imgsz=1280)
- RT-DETR-L : ~3~5시간 (150에폭, imgsz=1024)

**세션이 끊겨도 이어서 학습 가능** — 끊기면 해당 모델의 `RESUME = True` 로 바꾸고 셀 2 → 해당 모델 학습 셀 순서로 재실행  

> **사전 준비**: 런타임 → 런타임 유형 변경 → **GPU (A100 권장)**

## 셀 1. 설정 — **이 셀만 수정하세요**

| 구분 | 설명 |
|---|---|
| `DRIVE_PROJECT_DIR` | 구글 드라이브 프로젝트 루트 |
| `YOLO11_*` / `YOLO26_*` / `RTDETR_*` | 각 모델 학습 하이퍼파라미터 |
| `*_RESUME` | `True`면 Drive 백업 last.pt에서 이어서 학습 |
| `BACKUP_PERIOD` | N에폭마다 Drive로 가중치 백업 |
| `WBF_*` / `ENS_*` | 앙상블 설정 |
| `FINAL_SUBMISSION` | 최종 제출 CSV 파일명 |

In [ ]:
import os

# ═══════════════════════════════════════════════════════════════
#  여기만 수정하세요
# ═══════════════════════════════════════════════════════════════

# ── 드라이브 경로 ────────────────────────────────────────────────
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/베이스라인_코랩버전v1.4'

# ── YOLO11 학습 설정 ─────────────────────────────────────────────
YOLO11_MODEL   = 'yolo11x.pt'  # yolo11n / s / m / l / x
YOLO11_EPOCHS  = 60
YOLO11_BATCH   = 4
YOLO11_IMGSZ   = 1280
YOLO11_NAME    = 'yolo11'      # 저장 폴더 → outputs/yolo/yolo11/
YOLO11_RESUME  = False         # True: Drive last.pt에서 이어가기

# ── YOLO26 학습 설정 (앙상블 다양성용 두 번째 YOLO) ──────────────
YOLO26_MODEL   = 'yolo11x.pt'  # yolo11n / s / m / l / x
YOLO26_EPOCHS  = 60
YOLO26_BATCH   = 4
YOLO26_IMGSZ   = 1280
YOLO26_NAME    = 'yolo26'
YOLO26_RESUME  = False

# ── RT-DETR 학습 설정 ────────────────────────────────────────────
RTDETR_MODEL   = 'rtdetr-l.pt' # rtdetr-l.pt / rtdetr-x.pt
RTDETR_EPOCHS  = 150
RTDETR_BATCH   = 16            # A100: 16~24, T4: 8
RTDETR_IMGSZ   = 1024
RTDETR_NAME    = 'rtdetr'
RTDETR_RESUME  = False

# ── 공통 ─────────────────────────────────────────────────────────
BACKUP_PERIOD  = 10            # N 에폭마다 Drive 백업

# ── 앙상블 설정 ──────────────────────────────────────────────────
WBF_WEIGHTS      = '1 1 1'     # [yolo11  yolo26  rtdetr] 가중치
WBF_IOU          = 0.55
ENS_CONF         = 0.15
ENS_YOLO_IMGSZ   = 1280
ENS_MAX_DET      = 4

# ── 출력 파일명 ──────────────────────────────────────────────────
FINAL_SUBMISSION = 'submission_final.csv'

# ═══════════════════════════════════════════════════════════════
# 경로 자동 계산 (수정 불필요)
# ═══════════════════════════════════════════════════════════════
DRIVE_BASELINE    = os.path.join(DRIVE_PROJECT_DIR, 'baseline')
DRIVE_OUTPUTS     = os.path.join(DRIVE_PROJECT_DIR, 'outputs')
DRIVE_DATA_ZIP    = os.path.join(DRIVE_PROJECT_DIR, 'sprint_ai_project1_data.zip')
DRIVE_YOLO11_DIR  = os.path.join(DRIVE_OUTPUTS, 'yolo', YOLO11_NAME)
DRIVE_YOLO26_DIR  = os.path.join(DRIVE_OUTPUTS, 'yolo', YOLO26_NAME)
DRIVE_RTDETR_DIR  = os.path.join(DRIVE_OUTPUTS, 'yolo', RTDETR_NAME)

LOCAL_PROJECT_DIR = '/content/project'
LOCAL_BASELINE    = os.path.join(LOCAL_PROJECT_DIR, 'baseline')
LOCAL_OUTPUTS     = os.path.join(LOCAL_BASELINE, 'outputs')
LOCAL_DATA_DIR    = os.path.join(LOCAL_PROJECT_DIR, 'sprint_ai_project1_data')
LOCAL_YOLO11_PT   = f'outputs/yolo/{YOLO11_NAME}/weights/best.pt'
LOCAL_YOLO26_PT   = f'outputs/yolo/{YOLO26_NAME}/weights/best.pt'
LOCAL_RTDETR_PT   = f'outputs/yolo/{RTDETR_NAME}/weights/best.pt'

print('=== 설정 확인 ===')
print(f'Drive 프로젝트 : {DRIVE_PROJECT_DIR}')
print(f'  데이터 zip   : exists={os.path.exists(DRIVE_DATA_ZIP)}')
print(f'  baseline     : exists={os.path.exists(DRIVE_BASELINE)}')
print()
print(f'YOLO11  : {YOLO11_MODEL}  epochs={YOLO11_EPOCHS}  batch={YOLO11_BATCH}  imgsz={YOLO11_IMGSZ}  resume={YOLO11_RESUME}')
print(f'YOLO26  : {YOLO26_MODEL}  epochs={YOLO26_EPOCHS}  batch={YOLO26_BATCH}  imgsz={YOLO26_IMGSZ}  resume={YOLO26_RESUME}')
print(f'RT-DETR : {RTDETR_MODEL}  epochs={RTDETR_EPOCHS}  batch={RTDETR_BATCH}  imgsz={RTDETR_IMGSZ}  resume={RTDETR_RESUME}')
print()
print(f'앙상블  : weights=[{WBF_WEIGHTS}]  wbf_iou={WBF_IOU}  conf={ENS_CONF}  max_det={ENS_MAX_DET}')
print(f'출력    : outputs/predictions/{FINAL_SUBMISSION}')

## 셀 2. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 셀 3. Drive → Colab 로컬 복제

- `baseline/` 폴더 복사
- 데이터 zip 압축 해제
- 이미 있으면 skip (재실행 안전)

In [ ]:
import os, shutil, subprocess, time

os.makedirs(LOCAL_PROJECT_DIR, exist_ok=True)

# baseline 폴더
if os.path.exists(LOCAL_BASELINE):
    print('[skip] baseline: 이미 로컬에 있음')
elif not os.path.exists(DRIVE_BASELINE):
    raise FileNotFoundError(f'Drive에 baseline 폴더 없음: {DRIVE_BASELINE}')
else:
    t0 = time.time()
    shutil.copytree(DRIVE_BASELINE, LOCAL_BASELINE)
    print(f'[ok] baseline 복사 완료 ({time.time()-t0:.1f}s)')

# 데이터셋
if os.path.exists(LOCAL_DATA_DIR):
    print('[skip] dataset: 이미 해제됨')
else:
    if not os.path.exists(DRIVE_DATA_ZIP):
        raise FileNotFoundError(f'데이터 zip 없음: {DRIVE_DATA_ZIP}')
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    print('[..] dataset 해제 중 (수 분 소요)...')
    t0 = time.time()
    subprocess.run(['unzip', '-q', DRIVE_DATA_ZIP, '-d', LOCAL_DATA_DIR], check=True)
    print(f'[ok] dataset 해제 완료 ({time.time()-t0:.1f}s)')
    print('     하위:', sorted(os.listdir(LOCAL_DATA_DIR))[:5])

## 셀 4. 작업 디렉토리 + config 경로 갱신

In [ ]:
import os, yaml

os.chdir(LOCAL_BASELINE)
print('CWD:', os.getcwd())

cfg_path = 'configs/default.yaml'
try:
    with open(cfg_path, encoding='utf-8') as f:
        cfg = yaml.safe_load(f) or {}
except FileNotFoundError:
    cfg = {}

cfg.setdefault('data', {})
cfg['data'].update({
    'data_root':         LOCAL_DATA_DIR,
    'train_images':      cfg['data'].get('train_images', 'train_images'),
    'test_images':       cfg['data'].get('test_images', 'test_images'),
    'train_annotations': cfg['data'].get('train_annotations', 'train_annotations'),
    'processed_dir':     './data/processed',
    'val_ratio':         cfg['data'].get('val_ratio', 0.2),
    'num_workers':       2,
})
cfg.setdefault('output', {})
cfg['output'].setdefault('prediction_dir', './outputs/predictions')
cfg['output'].setdefault('checkpoint_dir', './outputs/checkpoints')

os.makedirs('configs', exist_ok=True)
with open(cfg_path, 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

print('data_root:', cfg['data']['data_root'])
print('[ok] configs/default.yaml 업데이트 완료')

## 셀 5. 패키지 설치

In [ ]:
!pip install -q -r requirements.txt
!pip install -q ultralytics ensemble-boxes

## 셀 6. GPU 확인

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  GPU 없음 → 런타임 유형을 GPU로 변경하세요')

## 셀 7. 어노테이션 전처리 (최초 1회)

In [ ]:
!python scripts/preprocess.py

## 셀 8. COCO → YOLO 형식 변환 (최초 1회)

세 모델이 같은 `dataset.yaml` / `orig_id_map.json`을 공유합니다.

In [ ]:
!python scripts/convert_to_yolo.py

---
## 셀 9. YOLO11 학습

- 모델: `YOLO11_MODEL` (기본 `yolo11x.pt`)
- 가중치 저장: `outputs/yolo/yolo11/weights/best.pt`
- **세션이 끊겼으면** 셀 1에서 `YOLO11_RESUME = True` → 이 셀 재실행

> Drive에 이미 완성된 `best.pt`가 있으면 아래 준비 셀이 자동으로 복사해 옵니다 (학습 셀 건너뛰어도 됨)

In [ ]:
# YOLO11 가중치 준비 (Drive 백업 → 로컬 복원)
import os, shutil

local_best = LOCAL_YOLO11_PT
local_last = f'outputs/yolo/{YOLO11_NAME}/weights/last.pt'
drive_best = os.path.join(DRIVE_YOLO11_DIR, 'weights', 'best.pt')
drive_last = os.path.join(DRIVE_YOLO11_DIR, 'weights', 'last.pt')

if os.path.exists(local_best):
    print(f'[skip] YOLO11 best.pt 이미 로컬에 있음 → 학습 셀 건너뛰어도 됩니다: {local_best}')
elif os.path.exists(drive_best) and not YOLO11_RESUME:
    os.makedirs(os.path.dirname(local_best), exist_ok=True)
    shutil.copy2(drive_best, local_best)
    print(f'[ok] Drive best.pt → 로컬 복사 완료 (학습 셀 건너뛰어도 됩니다)')
elif os.path.exists(drive_last) and YOLO11_RESUME:
    os.makedirs(os.path.dirname(local_last), exist_ok=True)
    shutil.copy2(drive_last, local_last)
    if os.path.exists(drive_best):
        shutil.copy2(drive_best, local_best)
    print('[ok] Drive last.pt 복원 완료 → 이어서 학습 준비됨')
else:
    print('[info] Drive 가중치 없음 → 처음부터 학습합니다')

In [ ]:
resume_flag = '--resume' if YOLO11_RESUME else ''

!python train_yolo.py \
    --model {YOLO11_MODEL} \
    --epochs {YOLO11_EPOCHS} \
    --batch {YOLO11_BATCH} \
    --imgsz {YOLO11_IMGSZ} \
    --output outputs/yolo \
    --name {YOLO11_NAME} \
    --backup_dir "{DRIVE_YOLO11_DIR}" \
    --backup_period {BACKUP_PERIOD} \
    {resume_flag}

---
## 셀 10. YOLO26 학습

- 모델: `YOLO26_MODEL` (기본 `yolo11x.pt`, YOLO11과 같은 아키텍처지만 독립 학습으로 앙상블 다양성 확보)
- 가중치 저장: `outputs/yolo/yolo26/weights/best.pt`
- **세션이 끊겼으면** 셀 1에서 `YOLO26_RESUME = True` → 이 셀 재실행

In [ ]:
# YOLO26 가중치 준비 (Drive 백업 → 로컬 복원)
import os, shutil

local_best = LOCAL_YOLO26_PT
local_last = f'outputs/yolo/{YOLO26_NAME}/weights/last.pt'
drive_best = os.path.join(DRIVE_YOLO26_DIR, 'weights', 'best.pt')
drive_last = os.path.join(DRIVE_YOLO26_DIR, 'weights', 'last.pt')

if os.path.exists(local_best):
    print(f'[skip] YOLO26 best.pt 이미 로컬에 있음 → 학습 셀 건너뛰어도 됩니다: {local_best}')
elif os.path.exists(drive_best) and not YOLO26_RESUME:
    os.makedirs(os.path.dirname(local_best), exist_ok=True)
    shutil.copy2(drive_best, local_best)
    print(f'[ok] Drive best.pt → 로컬 복사 완료 (학습 셀 건너뛰어도 됩니다)')
elif os.path.exists(drive_last) and YOLO26_RESUME:
    os.makedirs(os.path.dirname(local_last), exist_ok=True)
    shutil.copy2(drive_last, local_last)
    if os.path.exists(drive_best):
        shutil.copy2(drive_best, local_best)
    print('[ok] Drive last.pt 복원 완료 → 이어서 학습 준비됨')
else:
    print('[info] Drive 가중치 없음 → 처음부터 학습합니다')

In [ ]:
resume_flag = '--resume' if YOLO26_RESUME else ''

!python train_yolo26.py \
    --model {YOLO26_MODEL} \
    --epochs {YOLO26_EPOCHS} \
    --batch {YOLO26_BATCH} \
    --imgsz {YOLO26_IMGSZ} \
    --output outputs/yolo \
    --name {YOLO26_NAME} \
    --backup_dir "{DRIVE_YOLO26_DIR}" \
    --backup_period {BACKUP_PERIOD} \
    {resume_flag}

---
## 셀 11. RT-DETR 학습

- 모델: `RTDETR_MODEL` (기본 `rtdetr-l.pt`)
- 가중치 저장: `outputs/yolo/rtdetr/weights/best.pt`
- **세션이 끊겼으면** 셀 1에서 `RTDETR_RESUME = True` → 이 셀 재실행

In [ ]:
# RT-DETR 가중치 준비 (Drive 백업 → 로컬 복원)
import os, shutil

local_best = LOCAL_RTDETR_PT
local_last = f'outputs/yolo/{RTDETR_NAME}/weights/last.pt'
drive_best = os.path.join(DRIVE_RTDETR_DIR, 'weights', 'best.pt')
drive_last = os.path.join(DRIVE_RTDETR_DIR, 'weights', 'last.pt')

if os.path.exists(local_best):
    print(f'[skip] RT-DETR best.pt 이미 로컬에 있음 → 학습 셀 건너뛰어도 됩니다: {local_best}')
elif os.path.exists(drive_best) and not RTDETR_RESUME:
    os.makedirs(os.path.dirname(local_best), exist_ok=True)
    shutil.copy2(drive_best, local_best)
    print(f'[ok] Drive best.pt → 로컬 복사 완료 (학습 셀 건너뛰어도 됩니다)')
elif os.path.exists(drive_last) and RTDETR_RESUME:
    os.makedirs(os.path.dirname(local_last), exist_ok=True)
    shutil.copy2(drive_last, local_last)
    if os.path.exists(drive_best):
        shutil.copy2(drive_best, local_best)
    print('[ok] Drive last.pt 복원 완료 → 이어서 학습 준비됨')
else:
    print('[info] Drive 가중치 없음 → 처음부터 학습합니다')

In [ ]:
resume_flag = '--resume' if RTDETR_RESUME else ''

!python train_rtdetr.py \
    --model {RTDETR_MODEL} \
    --epochs {RTDETR_EPOCHS} \
    --batch {RTDETR_BATCH} \
    --imgsz {RTDETR_IMGSZ} \
    --patience 50 \
    --output outputs/yolo \
    --name {RTDETR_NAME} \
    --backup_dir "{DRIVE_RTDETR_DIR}" \
    --backup_period {BACKUP_PERIOD} \
    {resume_flag}

---
## 셀 12. 가중치 3개 확인

앙상블 전에 세 모델 가중치가 모두 있는지 체크합니다.

In [ ]:
import os

weights = {
    'YOLO11' : LOCAL_YOLO11_PT,
    'YOLO26' : LOCAL_YOLO26_PT,
    'RT-DETR': LOCAL_RTDETR_PT,
}

all_ok = True
for name, path in weights.items():
    exists = os.path.exists(path)
    status = '[ok]  ' if exists else '[없음]'
    print(f'{status} {name:8s}: {path}')
    if not exists:
        all_ok = False

if all_ok:
    print('\n세 모델 가중치 확인 완료 → 앙상블 셀 실행하세요')
else:
    print('\n[경고] 없는 가중치가 있습니다. 해당 모델 학습 셀을 먼저 실행하세요.')

---
## 셀 13. 3-모델 WBF 앙상블 추론 → `submission_final.csv`

YOLO11(1280+TTA) + YOLO26(1280+TTA) + RT-DETR(1024)을 WBF로 결합합니다.

In [ ]:
!python inference_ensemble_vomega.py \
    --yolo11 {LOCAL_YOLO11_PT} \
    --yolo26 {LOCAL_YOLO26_PT} \
    --rtdetr {LOCAL_RTDETR_PT} \
    --weights {WBF_WEIGHTS} \
    --yolo_imgsz {ENS_YOLO_IMGSZ} \
    --rtdetr_imgsz {RTDETR_IMGSZ} \
    --conf {ENS_CONF} \
    --iou 0.6 \
    --wbf_iou {WBF_IOU} \
    --max_det {ENS_MAX_DET} \
    --out outputs/predictions/{FINAL_SUBMISSION}

## 셀 14. 결과 확인

In [ ]:
import os, pandas as pd

path = f'outputs/predictions/{FINAL_SUBMISSION}'
if not os.path.exists(path):
    print(f'[없음] {path}')
else:
    df = pd.read_csv(path)
    print(f'=== {FINAL_SUBMISSION} ===')
    print(f'  총 예측 수 : {len(df)}')
    print(f'  이미지 수  : {df["image_id"].nunique()}')
    print(f'  클래스 수  : {df["category_id"].nunique()}')
    print(df.head(5).to_string(index=False))

## 셀 15. Drive 백업

`outputs/predictions/` (CSV)와 `outputs/yolo/` (세 모델 가중치)를 Drive로 저장합니다.

In [ ]:
import shutil, os, time

os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

for folder in ['predictions', 'yolo']:
    src = os.path.join(LOCAL_OUTPUTS, folder)
    dst = os.path.join(DRIVE_OUTPUTS, folder)
    if not os.path.exists(src):
        print(f'[skip] {folder}: 없음')
        continue
    t0 = time.time()
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'[ok] {folder} → Drive ({time.time()-t0:.1f}s)')

print('\n백업 완료:', DRIVE_OUTPUTS)
print('저장된 파일:')
for name, path in [('YOLO11', os.path.join(DRIVE_YOLO11_DIR, 'weights', 'best.pt')),
                   ('YOLO26', os.path.join(DRIVE_YOLO26_DIR, 'weights', 'best.pt')),
                   ('RT-DETR', os.path.join(DRIVE_RTDETR_DIR, 'weights', 'best.pt')),
                   ('제출 CSV', os.path.join(DRIVE_OUTPUTS, 'predictions', FINAL_SUBMISSION))]:
    print(f'  {name:10s}: exists={os.path.exists(path)}')